# Figure 6 Preparation

In [ ]:
# to create pt files with angles
# run code in "Figure 6 pt files creation" section
# in this notebook

In [ ]:
import torch
import numpy as np

In [2]:
def mean_std(l):
    return list(np.array(l)[:, 0]), list(np.array(l)[:, 1])

def quantitative(filename):
    similarity_results = torch.load(filename)
    intra_means, intra_stds = mean_std(similarity_results['intra_class_sims'])
    inter_means, inter_stds = mean_std(similarity_results['inter_class_sims'])
    noise_means, noise_stds = mean_std(similarity_results['class_vs_noise_sims'])

    def fmt(m, s):
        return f"{np.round(np.mean(m), 2)} ± {np.round(np.mean(s), 2)}"

    return {
        'i':   (np.mean(intra_means), np.mean(intra_stds)),
        'o':   (np.mean(inter_means), np.mean(inter_stds)),
        'b':   (np.mean(noise_means), np.mean(noise_stds)),
    }

In [4]:
def format_metrics(avg_metrics):
    labels = ['in', 'out', 'back']
    parts = []

    for label, v in zip(labels, avg_metrics.values()):
        mean = v[0]
        std = v[1]
        parts.append(f'{label} {mean:.2f}±{std:.2f}<br>')

    return ''.join(parts)

In [ ]:
s = f"""
| Audio | After Conv | After Shifting | $\Delta$ After Conv | $\Delta$ After Shifting |
| --- | --- | --- | --- | --- |
"""

path_dir = "heap"

for norm_type_name in [
    "mean_and_bias",
    "mean",
    "bias",
    "no_mean_no_bias",
]:
    cells = [norm_type_name]
    for time in ["after", "change"]:
        for operation in ["conv", "shift"]:
            filename = f"{path_dir}/{norm_type_name}_{time}_{operation}.pt"
            avg_metrics = quantitative(filename)
            cells.append(format_metrics(avg_metrics))
    
    s += f"| {' | '.join(cells)} |"
    s += "\n"

s = s[:-1].replace("no_mean_no_bias", "No Mean & No Bias").replace("mean_and_bias", "Mean & Bias").replace("mean", "Mean").replace("bias", "Bias")
print(s)

# Figure 8 Preparation

In [ ]:
# to create pt file with angles
# run code in "Figure 8 pt file creation" section
# in this notebook

In [ ]:
import re
import numpy as np
import torch


def mean_std(l):
    return list(np.array(l)[:, 0]), list(np.array(l)[:, 1])


res = torch.load("heap/fig8.pt")

series_keys = ['top_by_center', 'low_by_center', 'top_by_s', 'low_by_s']
labels      = ['Top by center', 'Low by center', 'Top by s', 'Low by s']
parsed = {s: {} for s in series_keys}
for k, v in res.items():
    m = re.match(r'(.+?)_([\d.]+)_norm$', k)
    if not m:
        continue
    series, thresh = m.group(1), float(m.group(2))
    if series not in parsed:
        continue
    means, stds = mean_std(v)
    parsed[series][thresh] = (np.mean(means), np.mean(stds))

x = sorted(next(iter(parsed.values())).keys())
d = {}
for series, label in zip(series_keys, labels):
    means = [parsed[series][xi][0] for xi in x]
    stds  = [parsed[series][xi][1] for xi in x]
    d[label] = (means, stds)
print(d)

# Figure 6 pt files creation

In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [4]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")

import torch

from my_help_functions.hooks import register_conv_bn_hooks
from my_help_functions.norms import (
    BiasWeightChangeBatchNormNothing,
    BiasWeightChangeBatchNormOnlyMean,
    BiasWeightChangeBatchNormOnlyBias,
    BiasWeightChangeBatchNorm
)
from my_help_functions.utils import calculate_and_save_angles_fig6, run_validation
from my_help_functions.prepare_model import prepare_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
manifest_path = 'val_manifest_w_noise.json'

In [5]:
configurations = {
    "Mean and Bias" : (BiasWeightChangeBatchNorm, "./weights/asr_model_mean_and_bias.ckpt"),
    "Mean" : (BiasWeightChangeBatchNormOnlyMean, "./weights/asr_model_mean.ckpt"),
    "Bias" : (BiasWeightChangeBatchNormOnlyBias, "./weights/asr_model_bias.ckpt"),
    "No Mean and No Bias" : (BiasWeightChangeBatchNormNothing, "./weights/asr_model_no_mean_no_bias.ckpt"),
}

In [6]:
# Choose any
conf = configurations["Mean and Bias"]

In [ ]:
asr_model = prepare_model(conf[1], conf[0], device)
bns, convs = register_conv_bn_hooks(asr_model, BiasWeightChangeBatchNorm)
# batch size has to be 8000, otherwise not all samples will be taken into account for angles
labels, predictions, accuracy = run_validation(asr_model, manifest_path, batch_size=8000)

In [ ]:
# will be saved 4 files:
# {save_dir}/{norm_type_name}_[after, change]_[conv, shift].pt
calculate_and_save_angles_fig6(convs, bns, labels, save_dir='heap', norm_type_name='mean_and_bias')

# Figure 8 pt file creation

In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"

In [4]:
os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [ ]:
import torch

from my_help_functions.hooks import register_hooks_corr_uncorr
from my_help_functions.norms import BiasWeightChangeBatchNorm
from my_help_functions.utils import calculate_and_save_angles_fig8, run_validation
from my_help_functions.prepare_model import prepare_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
manifest_path = 'val_manifest_w_noise.json'

In [ ]:
asr_model = prepare_model("./weights/asr_model_mean_and_bias.ckpt",
                          BiasWeightChangeBatchNorm,
                          device)

In [ ]:
res = {}
for v in [0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95]:
    for t in ['top', 'low']:
        for s in ['center', 's']:
            corr_type = f'{t}_by_{s}_{v}_norm'
            print(corr_type)
            convs = register_hooks_corr_uncorr(asr_model, corr_type, norm=BiasWeightChangeBatchNorm, with_s=True)
            labels, predictions, accuracy = run_validation(asr_model, manifest_path, batch_size=8000)
            res = calculate_and_save_angles_fig8(asr_model, convs, labels, res, corr_type)

torch.save(res, 'heap/fig8.pt')

# Test accuracy

In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"

In [4]:
os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [15]:
import torch
import lightning.pytorch as pl
from omegaconf import OmegaConf

from my_help_functions.hooks import register_hooks_corr_uncorr
from my_help_functions.norms import (
    BiasWeightChangeBatchNormNothing,
    BiasWeightChangeBatchNormOnlyMean,
    BiasWeightChangeBatchNormOnlyBias,
    BiasWeightChangeBatchNorm
)
from my_help_functions.utils import calculate_and_save_angles_fig8, run_validation
from my_help_functions.prepare_model import prepare_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
manifest_path = 'val_manifest_w_noise.json'

In [16]:
configurations = {
    "Mean and Bias" : (BiasWeightChangeBatchNorm, "./weights/asr_model_mean_and_bias.ckpt"),
    "Mean" : (BiasWeightChangeBatchNormOnlyMean, "./weights/asr_model_mean.ckpt"),
    "Bias" : (BiasWeightChangeBatchNormOnlyBias, "./weights/asr_model_bias.ckpt"),
    "No Mean and No Bias" : (BiasWeightChangeBatchNormNothing, "./weights/asr_model_no_mean_no_bias.ckpt"),
}

In [20]:
# Choose any
conf = configurations["Mean and Bias"]

In [ ]:
asr_model = prepare_model(conf[1], conf[0], device)

In [10]:
data_dir = './sound_dataset_demo/'
MODEL_CONFIG = "matchboxnet_3x2x64_v1.yaml"
dataset_path = 'google_speech_recognition_v1'
dataset_basedir = os.path.join(data_dir, dataset_path)
config_path = f"configs/{MODEL_CONFIG}"
config = OmegaConf.load(config_path)
config = OmegaConf.to_container(config, resolve=True)
config = OmegaConf.create(config)

In [ ]:
accelerator = 'gpu'
config.trainer.devices = 1
config.trainer.num_nodes = 1
config.trainer.accelerator = accelerator
config.trainer.strategy = 'auto'

print("Trainer config - \n")
print(OmegaConf.to_yaml(config.trainer))

trainer = pl.Trainer(precision=16, enable_progress_bar=False, **config.trainer)

In [ ]:
asr_model.setup_test_data(
    test_data_config={
        'manifest_filepath': os.path.join(dataset_basedir, 'test_manifest.json'),
        'sample_rate': 16000,
        'labels': asr_model.cfg.labels,
        'batch_size': 4096,
        'shuffle': False,
        'num_workers': 4
    }
)
trainer.test(asr_model, ckpt_path=None)